In [2]:
## Plots old cross sectional momentum

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import numpy as np
import math

def cross_sectional_momentum(data, lookback_period, holding_period, top_n_stocks):
    """
    Implement cross-sectional momentum strategy for top n stocks by market cap.
    
    :param data: DataFrame with columns 'date', 'ticker', 'PX_LAST', 'CUR_MKT_CAP'
    :param lookback_period: int, number of days for lookback period
    :param holding_period: int, number of days for holding period
    :param top_n_stocks: int, number of top market cap stocks to consider
    :return: DataFrame with top stocks for each period
    """
    # Get the latest market cap for each stock
    latest_market_cap = data.groupby('ticker')['CUR_MKT_CAP'].last().sort_values(ascending=False)
    top_stocks = latest_market_cap.head(top_n_stocks).index.tolist()

    # Filter data for top n stocks
    data = data[data['ticker'].isin(top_stocks)]

    # Sort data by date and ticker
    data = data.sort_values(['date', 'ticker'])
    
    # Group by ticker and calculate returns
    returns = data.groupby('ticker').apply(lambda x: x.set_index('date')['PX_LAST'].pct_change(periods=lookback_period))
    returns = returns.reset_index()
    returns.columns = ['ticker', 'date', 'return']
    
    # Function to get top stocks for each date
    def get_top_stocks(group):
        return group.nlargest(min(len(top_stocks), 5), 'return')
    
    # Group by date and get top stocks
    top_performing_stocks = returns.groupby('date').apply(get_top_stocks).reset_index(drop=True)
    
    # Filter dates to match holding period
    unique_dates = top_performing_stocks['date'].unique()
    filtered_dates = unique_dates[::holding_period]
    
    # Filter top stocks for the selected dates
    top_stocks_filtered = top_performing_stocks[top_performing_stocks['date'].isin(filtered_dates)]
    
    return top_stocks_filtered

def plot_top_stocks(data, results, lookback_period, holding_period, stocks_per_plot=5):
    """
    Plot the top stocks identified by the cross-sectional momentum strategy.
    
    :param data: DataFrame with columns 'date', 'ticker', 'PX_LAST'
    :param results: DataFrame with results from cross_sectional_momentum function
    :param lookback_period: int, number of days for lookback period
    :param holding_period: int, number of days for holding period
    :param stocks_per_plot: int, number of stocks to plot per figure
    """
    # Get unique tickers from results
    unique_tickers = results['ticker'].unique()
    
    # Calculate number of plots needed
    num_plots = math.ceil(len(unique_tickers) / stocks_per_plot)
    
    for plot_num in range(num_plots):
        # Select tickers for this plot
        plot_tickers = unique_tickers[plot_num*stocks_per_plot:(plot_num+1)*stocks_per_plot]
        
        # Set up the plot
        fig, axs = plt.subplots(len(plot_tickers), 1, figsize=(15, 5*len(plot_tickers)), sharex=True)
        fig.suptitle(f'Top Stocks Identified by Cross-Sectional Momentum Strategy (Page {plot_num+1})', fontsize=16)
        
        axs = axs if len(plot_tickers) > 1 else [axs]  # Ensure axs is always a list
        
        for idx, ticker in enumerate(plot_tickers):
            # Filter data for this ticker
            stock_data = data[data['ticker'] == ticker]
            stock_results = results[results['ticker'] == ticker]
            
            # Plot stock price
            axs[idx].plot(stock_data['date'], stock_data['PX_LAST'], label='Stock Price', color='blue')
            
            # Highlight periods when the stock was selected
            for _, row in stock_results.iterrows():
                start_date = row['date'] - pd.Timedelta(days=lookback_period)
                end_date = row['date'] + pd.Timedelta(days=holding_period)
                axs[idx].axvspan(start_date, end_date, color='green', alpha=0.3)
            
            # Set labels and title
            axs[idx].set_ylabel('Price')
            axs[idx].set_title(f'{ticker} Stock Price')
            axs[idx].legend()
            
            # Format x-axis
            axs[idx].xaxis.set_major_formatter(DateFormatter('%Y-%m-%d'))
        
        plt.xlabel('Date')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

# Example usage
lookback_period = 126  # Approximately 6 months of trading days
holding_period = 21   # Approximately 1 month of trading days
top_n_stocks = 50  # Number of top market cap stocks to consider

# Get results from cross-sectional momentum function
results = cross_sectional_momentum(df, lookback_period, holding_period, top_n_stocks)

# Plot the results
plot_top_stocks(df, results, lookback_period, holding_period, stocks_per_plot=5)

In [ ]:
## Cross-sectional v2

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import numpy as np
import math

def cross_sectional_momentum(data, lookback_period, holding_period):
    """
    Implement cross-sectional momentum strategy.
    
    :param data: DataFrame with columns 'date', 'ticker', 'PX_LAST'
    :param lookback_period: int, number of days for lookback period
    :param holding_period: int, number of days for holding period
    :return: DataFrame with top 5 stocks for each period
    """
    # Sort data by date and ticker
    data = data.sort_values(['date', 'ticker'])
    
    # Group by ticker and calculate returns
    returns = data.groupby('ticker').apply(lambda x: x.set_index('date')['PX_LAST'].pct_change(periods=lookback_period))
    returns = returns.reset_index()
    returns.columns = ['ticker', 'date', 'return']
    
    # Function to get top 5 stocks for each date
    def get_top_5(group):
        return group.nlargest(5, 'return')
    
    # Group by date and get top 5 stocks
    top_stocks = returns.groupby('date').apply(get_top_5).reset_index(drop=True)
    
    # Filter dates to match holding period
    unique_dates = top_stocks['date'].unique()
    filtered_dates = unique_dates[::holding_period]
    
    # Filter top stocks for the selected dates
    top_stocks_filtered = top_stocks[top_stocks['date'].isin(filtered_dates)]
    
    return top_stocks_filtered

def plot_top_stocks(data, results, lookback_period, holding_period, stocks_per_plot=5):
    """
    Plot the top stocks identified by the cross-sectional momentum strategy.
    
    :param data: DataFrame with columns 'date', 'ticker', 'PX_LAST'
    :param results: DataFrame with results from cross_sectional_momentum function
    :param lookback_period: int, number of days for lookback period
    :param holding_period: int, number of days for holding period
    :param stocks_per_plot: int, number of stocks to plot per figure
    """
    # Get unique tickers from results
    unique_tickers = results['ticker'].unique()
    
    # Calculate number of plots needed
    num_plots = math.ceil(len(unique_tickers) / stocks_per_plot)
    
    for plot_num in range(num_plots):
        # Select tickers for this plot
        plot_tickers = unique_tickers[plot_num*stocks_per_plot:(plot_num+1)*stocks_per_plot]
        
        # Set up the plot
        fig, axs = plt.subplots(len(plot_tickers), 1, figsize=(15, 5*len(plot_tickers)), sharex=True)
        fig.suptitle(f'Top Stocks Identified by Cross-Sectional Momentum Strategy (Page {plot_num+1})', fontsize=16)
        
        for idx, ticker in enumerate(plot_tickers):
            # Filter data for this ticker
            stock_data = data[data['ticker'] == ticker]
            stock_results = results[results['ticker'] == ticker]
            
            # Plot stock price
            axs[idx].plot(stock_data['date'], stock_data['PX_LAST'], label='Stock Price', color='blue')
            
            # Highlight periods when the stock was selected
            for _, row in stock_results.iterrows():
                start_date = row['date'] - pd.Timedelta(days=lookback_period)
                end_date = row['date'] + pd.Timedelta(days=holding_period)
                axs[idx].axvspan(start_date, end_date, color='green', alpha=0.3)
            
            # Set labels and title
            axs[idx].set_ylabel('Price')
            axs[idx].set_title(f'{ticker} Stock Price')
            axs[idx].legend()
            
            # Format x-axis
            axs[idx].xaxis.set_major_formatter(DateFormatter('%Y-%m-%d'))
        
        plt.xlabel('Date')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

# Your DataFrame is already loaded as 'df'

# Example usage
# lookback_period = 126  # Approximately 6 months of trading days
# holding_period = 21   # Approximately 1 month of trading days

# # Get results from cross-sectional momentum function
# results = cross_sectional_momentum(df, lookback_period, holding_period)

# # Plot the results
# plot_top_stocks(df, results, lookback_period, holding_period, stocks_per_plot=5)

In [ ]:
## cross sectional momentum v3

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter
import numpy as np
import math

def cross_sectional_momentum(data, lookback_period, holding_period, top_n_stocks):
    """
    Implement cross-sectional momentum strategy for top n stocks by market cap.
    
    :param data: DataFrame with columns 'date', 'ticker', 'PX_LAST', 'CUR_MKT_CAP'
    :param lookback_period: int, number of days for lookback period
    :param holding_period: int, number of days for holding period
    :param top_n_stocks: int, number of top market cap stocks to consider
    :return: DataFrame with top stocks for each period
    """
    # Get the latest market cap for each stock
    latest_market_cap = data.groupby('ticker')['CUR_MKT_CAP'].last().sort_values(ascending=False)
    top_stocks = latest_market_cap.head(top_n_stocks).index.tolist()

    # Filter data for top n stocks
    data = data[data['ticker'].isin(top_stocks)]

    # Sort data by date and ticker
    data = data.sort_values(['date', 'ticker'])
    
    # Group by ticker and calculate returns
    returns = data.groupby('ticker').apply(lambda x: x.set_index('date')['PX_LAST'].pct_change(periods=lookback_period))
    returns = returns.reset_index()
    returns.columns = ['ticker', 'date', 'return']
    
    # Function to get top stocks for each date
    def get_top_stocks(group):
        return group.nlargest(min(len(top_stocks), 5), 'return')
    
    # Group by date and get top stocks
    top_performing_stocks = returns.groupby('date').apply(get_top_stocks).reset_index(drop=True)
    
    # Filter dates to match holding period
    unique_dates = top_performing_stocks['date'].unique()
    filtered_dates = unique_dates[::holding_period]
    
    # Filter top stocks for the selected dates
    top_stocks_filtered = top_performing_stocks[top_performing_stocks['date'].isin(filtered_dates)]
    
    return top_stocks_filtered

def plot_top_stocks(data, results, lookback_period, holding_period, stocks_per_plot=5):
    """
    Plot the top stocks identified by the cross-sectional momentum strategy.
    
    :param data: DataFrame with columns 'date', 'ticker', 'PX_LAST'
    :param results: DataFrame with results from cross_sectional_momentum function
    :param lookback_period: int, number of days for lookback period
    :param holding_period: int, number of days for holding period
    :param stocks_per_plot: int, number of stocks to plot per figure
    """
    # Get unique tickers from results
    unique_tickers = results['ticker'].unique()
    
    # Calculate number of plots needed
    num_plots = math.ceil(len(unique_tickers) / stocks_per_plot)
    
    for plot_num in range(num_plots):
        # Select tickers for this plot
        plot_tickers = unique_tickers[plot_num*stocks_per_plot:(plot_num+1)*stocks_per_plot]
        
        # Set up the plot
        fig, axs = plt.subplots(len(plot_tickers), 1, figsize=(15, 5*len(plot_tickers)), sharex=True)
        fig.suptitle(f'Top Stocks Identified by Cross-Sectional Momentum Strategy (Page {plot_num+1})', fontsize=16)
        
        axs = axs if len(plot_tickers) > 1 else [axs]  # Ensure axs is always a list
        
        for idx, ticker in enumerate(plot_tickers):
            # Filter data for this ticker
            stock_data = data[data['ticker'] == ticker]
            stock_results = results[results['ticker'] == ticker]
            
            # Plot stock price
            axs[idx].plot(stock_data['date'], stock_data['PX_LAST'], label='Stock Price', color='blue')
            
            # Highlight periods when the stock was selected
            for _, row in stock_results.iterrows():
                start_date = row['date'] - pd.Timedelta(days=lookback_period)
                end_date = row['date'] + pd.Timedelta(days=holding_period)
                axs[idx].axvspan(start_date, end_date, color='green', alpha=0.3)
            
            # Set labels and title
            axs[idx].set_ylabel('Price')
            axs[idx].set_title(f'{ticker} Stock Price')
            axs[idx].legend()
            
            # Format x-axis
            axs[idx].xaxis.set_major_formatter(DateFormatter('%Y-%m-%d'))
        
        plt.xlabel('Date')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

# Example usage
lookback_period = 126  # Approximately 6 months of trading days
holding_period = 21   # Approximately 1 month of trading days
top_n_stocks = 10  # Number of top market cap stocks to consider

# Get results from cross-sectional momentum function
results = cross_sectional_momentum(unique_dates_per_ticker_df, lookback_period, holding_period, top_n_stocks)

# Plot the results
plot_top_stocks(unique_dates_per_ticker_df, results, lookback_period, holding_period, stocks_per_plot=5)

In [3]:
### Plots the ARIMA forecast

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.tseries.offsets import QuarterEnd
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima

def analyze_and_forecast_stocks(data, num_stocks, forecast_quarters):
    # Ensure date is in datetime format
    data['date'] = pd.to_datetime(data['date'])
    
    # Get the latest market cap for each stock and sort
    latest_market_cap = data.groupby('ticker')['CUR_MKT_CAP'].last().sort_values(ascending=False)
    top_stocks = latest_market_cap.head(num_stocks).index.tolist()

    def format_date_with_quarter(date):
        year = date.year
        quarter = (date.month - 1) // 3 + 1
        return f"{year}-Q{quarter}"

    for ticker in top_stocks:
        print(f"\nAnalyzing {ticker}")
        
        # Filter data for this ticker
        stock_data = data[data['ticker'] == ticker].sort_values('date')
        
        # Resample to quarterly data
        quarterly_data = stock_data.set_index('date')['PX_LAST'].resample('Q').last().dropna()
        
        if len(quarterly_data) < 8:
            print(f"Not enough data for {ticker}. Skipping.")
            continue

        # Calculate Simple Moving Average (SMA)
        ma_window = 4  # 1-year moving average
        sma = quarterly_data.rolling(window=ma_window).mean()

        # Calculate Exponential Moving Average (EMA)
        ema = quarterly_data.ewm(span=ma_window, adjust=False).mean()

        # ARIMA Forecasting
        try:
            # Automatically select ARIMA parameters
            auto_model = auto_arima(quarterly_data, seasonal=True, m=4, suppress_warnings=True, error_action="ignore")
            
            # Fit the ARIMA model
            model = ARIMA(quarterly_data, order=auto_model.order, seasonal_order=auto_model.seasonal_order)
            results = model.fit()

            # Generate forecast
            forecast = results.forecast(steps=forecast_quarters)
            forecast_dates = pd.date_range(start=quarterly_data.index[-1] + QuarterEnd(), periods=forecast_quarters, freq='Q')
            forecast.index = forecast_dates

            # Plot results
            plt.figure(figsize=(12, 6))
            plt.plot(quarterly_data.index, quarterly_data.values, label='Historical Data')
            plt.plot(sma.index, sma.values, label='Simple Moving Average', color='orange')
            plt.plot(ema.index, ema.values, label='Exponential Moving Average', color='green')
            plt.plot(forecast.index, forecast.values, label='ARIMA Forecast', color='red')
            plt.title(f'{ticker} Stock Price, Moving Averages, and ARIMA Forecast')
            plt.xlabel('Date')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True)
            plt.show()

            # Print forecast values
            print(f"\nARIMA Forecast for {ticker}:")
            for date, price in forecast.items():
                formatted_date = format_date_with_quarter(date)
                print(f"{formatted_date}: ${price:.2f}")

            # Print some statistics
            print(f"Data range: ${quarterly_data.min():.2f} to ${quarterly_data.max():.2f}")
            print(f"Number of quarters: {len(quarterly_data)}")
            print(f"Current price: ${quarterly_data.iloc[-1]:.2f}")
            print(f"Current SMA: ${sma.iloc[-1]:.2f}")
            print(f"Current EMA: ${ema.iloc[-1]:.2f}")
            print(f"ARIMA Model: {auto_model.order}, Seasonal: {auto_model.seasonal_order}")

        except Exception as e:
            print(f"Error in ARIMA forecasting for {ticker}: {str(e)}")

# Example usage
num_stocks_to_forecast = 5
quarters_to_forecast = 10

# Assuming your DataFrame is named 'unique_dates_per_ticker_df'
analyze_and_forecast_stocks(unique_dates_per_ticker_df, num_stocks_to_forecast, quarters_to_forecast)

/Users/biryani/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


NameError: name 'unique_dates_per_ticker_df' is not defined

In [ ]:
def analyze_and_forecast_stocks(data, df_2024, num_stocks, forecast_quarters):
    # Ensure date is in datetime format for both dataframes
    data['date'] = pd.to_datetime(data['date'])
    df_2024['date'] = pd.to_datetime(df_2024['date'])
    
    # Get the latest market cap for each stock and sort
    latest_market_cap = data.groupby('ticker')['CUR_MKT_CAP'].last().sort_values(ascending=False)
    top_stocks = latest_market_cap.head(num_stocks).index.tolist()

    def format_date_with_quarter(date):
        year = date.year
        quarter = (date.month - 1) // 3 + 1
        return f"{year}-Q{quarter}"

    for ticker in top_stocks:
        print(f"\nAnalyzing {ticker}")
        
        # Filter data for this ticker
        stock_data = data[data['ticker'] == ticker].sort_values('date')
        stock_data_2024 = df_2024[df_2024['ticker'] == ticker].sort_values('date')
        
        # Resample to quarterly data
        quarterly_data = stock_data.set_index('date')['PX_LAST'].resample('Q').last().dropna()
        quarterly_data_2024 = stock_data_2024.set_index('date')['PX_LAST'].resample('Q').last().dropna()
        
        if len(quarterly_data) < 8:
            print(f"Not enough data for {ticker}. Skipping.")
            continue

        # Calculate Simple Moving Average (SMA)
        ma_window = 4  # 1-year moving average
        sma = quarterly_data.rolling(window=ma_window).mean()

        # Calculate Exponential Moving Average (EMA)
        ema = quarterly_data.ewm(span=ma_window, adjust=False).mean()

        # ARIMA Forecasting
        try:
            # Automatically select ARIMA parameters
            auto_model = auto_arima(quarterly_data, seasonal=True, m=4, suppress_warnings=True, error_action="ignore")
            
            # Fit the ARIMA model
            model = ARIMA(quarterly_data, order=auto_model.order, seasonal_order=auto_model.seasonal_order)
            results = model.fit()

            # Generate forecast
            forecast = results.forecast(steps=forecast_quarters)
            forecast_dates = pd.date_range(start=quarterly_data.index[-1] + QuarterEnd(), periods=forecast_quarters, freq='Q')
            forecast.index = forecast_dates

            # Plot results
            plt.figure(figsize=(12, 6))
            plt.plot(quarterly_data.index, quarterly_data.values, label='Historical Data (2000-2022)')
            plt.plot(quarterly_data_2024.index, quarterly_data_2024.values, label='Actual Data (2023-2024)', color='purple')
            plt.plot(sma.index, sma.values, label='Simple Moving Average', color='orange')
            plt.plot(ema.index, ema.values, label='Exponential Moving Average', color='green')
            plt.plot(forecast.index, forecast.values, label='ARIMA Forecast', color='red')
            plt.title(f'{ticker} Stock Price, Moving Averages, and ARIMA Forecast')
            plt.xlabel('Date')
            plt.ylabel('Price')
            plt.legend()
            plt.grid(True)
            plt.show()

            # Print forecast values
            print(f"\nARIMA Forecast for {ticker}:")
            for date, price in forecast.items():
                formatted_date = format_date_with_quarter(date)
                print(f"{formatted_date}: ${price:.2f}")

            # Print some statistics
            print(f"Data range: ${quarterly_data.min():.2f} to ${quarterly_data.max():.2f}")
            print(f"Number of quarters: {len(quarterly_data)}")
            print(f"Current price (2022): ${quarterly_data.iloc[-1]:.2f}")
            print(f"Latest price (2024): ${quarterly_data_2024.iloc[-1]:.2f}")
            print(f"Current SMA: ${sma.iloc[-1]:.2f}")
            print(f"Current EMA: ${ema.iloc[-1]:.2f}")
            print(f"ARIMA Model: {auto_model.order}, Seasonal: {auto_model.seasonal_order}")

        except Exception as e:
            print(f"Error in ARIMA forecasting for {ticker}: {str(e)}")

analyze_and_forecast_stocks(df_2022, df_2024, num_stocks=5, forecast_quarters=4)